# Reproducing the accepted model

This notebook invokes the same canonical command-line workflow documented in the root `README.md`. It does not run the legacy pipeline or Gaussian calculations. The notebook is distributed without execution outputs; start the kernel from the repository root.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / "libs" / "current_model.py").is_file(), (
    "Start this notebook from the repository root."
)
WORKERS = min(4, os.cpu_count() or 1)  # Adjust within the supported range of 1-20.
RUN_ENV = os.environ.copy()
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    RUN_ENV[variable] = "1"

def run_cli(*arguments: str) -> None:
    subprocess.run(
        [sys.executable, *arguments],
        cwd=REPO_ROOT,
        env=RUN_ENV,
        check=True,
    )

## 1. Verify immutable inputs

Check the manifest byte sizes and SHA-256 values, together with the portable input schemas.

In [ ]:
run_cli("libs/current_model.py", "--verify-inputs-only")

## 2. Recompute strict nested LOOCV

Recompute all 83 outer folds without `--skip-nested`. Only the optional, Git-excluded contribution cubes are omitted.

In [ ]:
run_cli(
    "libs/current_model.py",
    "--workers",
    str(WORKERS),
    "--no-excel-refresh",
    "--skip-contribution-cubes",
)

## 3. Recompute the spatial analysis

Reconstruct coefficients for all 83 outer models and update the spatial-effect tables and reference figures.

In [ ]:
run_cli(
    "libs/analyze_current_model_spatial_contributions.py",
    "--workers",
    str(WORKERS),
    "--no-excel-refresh",
)

## 4. Verify saved results

Independently verify the frozen package, the 161 x 321 feature matrix, `results/model/summary.csv`, and metrics recalculated from all 83 outer predictions.

In [ ]:
run_cli("scripts/verify_reproduction.py")